# 01 - CheXpert Data Exploration

This notebook performs exploratory data analysis on the CheXpert dataset:
- Label distributions (positive / negative / uncertain / unmentioned)
- Co-occurrence of pathologies
- Frontal vs lateral view breakdown
- Sample image visualisation

In [ ]:
# Mount Google Drive (run this cell on Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Not running in Colab -- using local paths.')

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image

sns.set_theme(style='whitegrid')

# Add project root to path so we can import src modules
if IN_COLAB:
    PROJECT_ROOT = '/content/drive/MyDrive/Research Project'
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

sys.path.insert(0, PROJECT_ROOT)

In [ ]:
from src.utils import load_config

config = load_config(os.path.join(PROJECT_ROOT, 'configs', 'default.yaml'))
DATA_DIR = config['data']['data_dir']

train_csv = os.path.join(DATA_DIR, config['data']['train_csv'])
valid_csv = os.path.join(DATA_DIR, config['data']['valid_csv'])

print(f'Data directory: {DATA_DIR}')
print(f'Train CSV:      {train_csv}')
print(f'Valid CSV:      {valid_csv}')

In [ ]:
df_train = pd.read_csv(train_csv)
df_valid = pd.read_csv(valid_csv)

print(f'Train: {len(df_train):,} images')
print(f'Valid: {len(df_valid):,} images')
df_train.head()

## Label Columns

The 14 pathology labels plus their encoding:
- `1.0` = positive
- `0.0` = negative
- `-1.0` = uncertain
- `NaN` = unmentioned

In [ ]:
ALL_LABELS = [
    'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly',
    'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation',
    'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion',
    'Pleural Other', 'Fracture', 'Support Devices',
]

TARGET_LABELS = config['data']['target_labels']
print('Competition tasks:', TARGET_LABELS)

## 1. Label Distribution per Pathology

In [ ]:
def count_label_types(df, labels):
    """Count positive, negative, uncertain, and unmentioned per label column."""
    rows = []
    for col in labels:
        s = df[col]
        rows.append({
            'Pathology': col,
            'Positive (1)': (s == 1.0).sum(),
            'Negative (0)': (s == 0.0).sum(),
            'Uncertain (-1)': (s == -1.0).sum(),
            'Unmentioned (NaN)': s.isna().sum(),
        })
    return pd.DataFrame(rows).set_index('Pathology')

counts = count_label_types(df_train, ALL_LABELS)
counts

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
counts.plot(kind='barh', stacked=True, ax=ax,
            color=['#2ecc71', '#e74c3c', '#f39c12', '#95a5a6'])
ax.set_xlabel('Count')
ax.set_title('Label Distribution Across All 14 Pathologies (Train Set)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

### Focus on Competition Tasks (5 pathologies)

In [ ]:
target_counts = count_label_types(df_train, TARGET_LABELS)

fig, ax = plt.subplots(figsize=(10, 4))
target_counts.plot(kind='barh', stacked=True, ax=ax,
                   color=['#2ecc71', '#e74c3c', '#f39c12', '#95a5a6'])
ax.set_xlabel('Count')
ax.set_title('Label Distribution - 5 Competition Pathologies (Train Set)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

print('\nUncertain label percentages:')
for col in TARGET_LABELS:
    total = df_train[col].notna().sum()
    uncertain = (df_train[col] == -1.0).sum()
    print(f'  {col:25s}: {uncertain:6d} / {total:6d}  ({100*uncertain/total:.1f}%)')

## 2. Pathology Co-occurrence Matrix

In [ ]:
binary = (df_train[TARGET_LABELS] == 1.0).astype(int)
cooccurrence = binary.T.dot(binary)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cooccurrence, annot=True, fmt='d', cmap='YlOrRd', ax=ax)
ax.set_title('Positive Label Co-occurrence (Train Set)')
plt.tight_layout()
plt.show()

## 3. Frontal vs Lateral View

In [ ]:
if 'Frontal/Lateral' in df_train.columns:
    view_counts = df_train['Frontal/Lateral'].value_counts()
    print('View distribution (train):')
    print(view_counts)
    print(f'\nFrontal percentage: {100 * view_counts.get("Frontal", 0) / len(df_train):.1f}%')
else:
    print('Frontal/Lateral column not found.')

## 4. Sample Images

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Use parent of DATA_DIR as root for image paths
img_root = os.path.dirname(DATA_DIR)

sample = df_train.sample(8, random_state=42)
for ax, (_, row) in zip(axes.flat, sample.iterrows()):
    img_path = os.path.join(img_root, row['Path'])
    try:
        img = Image.open(img_path)
        ax.imshow(img, cmap='gray')
    except FileNotFoundError:
        ax.text(0.5, 0.5, 'Image not found', ha='center', va='center',
                transform=ax.transAxes)

    # Build a short title from positive labels
    pos = [l for l in TARGET_LABELS if row.get(l) == 1.0]
    unc = [l for l in TARGET_LABELS if row.get(l) == -1.0]
    title = ', '.join(pos) if pos else 'None positive'
    if unc:
        title += f'\nUncertain: {", ".join(unc)}'
    ax.set_title(title, fontsize=8)
    ax.axis('off')

fig.suptitle('Sample Chest X-Ray Images', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Validation Set Summary

In [ ]:
val_counts = count_label_types(df_valid, TARGET_LABELS)
print('Validation set label counts:')
val_counts